In [1]:
import os
import glob
import numpy as np
import xarray as xr
import pandas as pd
import dask
import dask.array as da
from datetime import timezone, timedelta

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from dask.distributed import wait

import time

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
client = Client(n_workers=18,
    threads_per_worker=1,
    memory_limit=f"{int(7)}GB"
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 18
Total threads: 18,Total memory: 117.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42243,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34487,Total threads: 1
Dashboard: /proxy/42285/status,Memory: 6.52 GiB
Nanny: tcp://127.0.0.1:33439,


These define whether which sample we're calcualting and which model we're using.

In [3]:
reanalysis = 'BARRA-C2'

In [4]:
hw_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")
bl_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv")

if reanalysis == 'BARRA-C2':
    extent = [147.5, 151, -38.5, -33.5]
elif reanalysis == 'BARRA-R2':
    extent = None

lon_min, lon_max, lat_min, lat_max = extent

In [5]:
# Statistically significant w ssmin=20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'CRURWF1',
            'WOODLWN1',
            'BOCORWF1',
            'BODWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [6]:
def get_days(days_df, nc_dir, extent=None):
    
    date_list = pd.to_datetime(days_df['date']).unique()
    year_months = {d.strftime("%Y%m") for d in date_list}
    patterns = [os.path.join(nc_dir, f"*{ym}*") for ym in year_months]
    selected_files = sorted(list(set(f for p in patterns for f in glob.glob(p))))
    if not selected_files:
        print(f"WARNING: No files found in {nc_dir}.")

    # Test one file to find indices for isel, no need to read coords
    lon_min, lon_max, lat_min, lat_max = extent
    
    with xr.open_dataset(selected_files[0], engine="netcdf4") as ds_single:
        lons = ds_single["lon"].values
        lats = ds_single["lat"].values

    lon_min_idx = np.searchsorted(lons, lon_min, side="left")
    lon_max_idx = np.searchsorted(lons, lon_max, side="right") - 1
    lat_min_idx = np.searchsorted(lats, lat_min, side="left")
    lat_max_idx = np.searchsorted(lats, lat_max, side="right") - 1

    print(f"INFO: Opening {len(selected_files)} files with xarray/dask...")
    ds = xr.open_mfdataset(
    selected_files,
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    chunks="auto",  # Use native on-disk chunking
    engine='netcdf4'
    )

    ds = ds.chunk({'time':80, 'lat':-1,'lon':-1})
    
    ds = ds.isel(
        lon=slice(lon_min_idx, lon_max_idx),
        lat=slice(lat_min_idx, lat_max_idx)
    )
    
    return ds

In [7]:
def get_days_delayed(days_df, nc_dir, extent=None):
    date_list = pd.to_datetime(days_df['date']).unique()
    year_months = {d.strftime("%Y%m") for d in date_list}
    patterns = [os.path.join(nc_dir, f"*{ym}*") for ym in year_months]
    selected_files = sorted(list(set(f for p in patterns for f in glob.glob(p))))
    if not selected_files:
        print(f"WARNING: No files found in {nc_dir}.")

    lon_min, lon_max, lat_min, lat_max = extent

    # Find indices for one file (to avoid loading all files for coords)
    with xr.open_dataset(selected_files[0], engine="netcdf4") as ds_single:
        lons = ds_single["lon"].values
        lats = ds_single["lat"].values

    lon_min_idx = np.searchsorted(lons, lon_min, side="left")
    lon_max_idx = np.searchsorted(lons, lon_max, side="right") - 1
    lat_min_idx = np.searchsorted(lats, lat_min, side="left")
    lat_max_idx = np.searchsorted(lats, lat_max, side="right") - 1
    
    @dask.delayed
    def open_and_crop(file):
        ds = xr.open_dataset(file, engine="netcdf4", chunks="auto")
        ds = ds.chunk({'time': 80, 'lat': -1, 'lon': -1})
        
        # Only select the lat/lon subset for each file
        ds = ds.isel(
            lon=slice(lon_min_idx, lon_max_idx),
            lat=slice(lat_min_idx, lat_max_idx)
        )
        return ds

    print('Building list of delayed tasks...')
    delayed_datasets = [open_and_crop(f) for f in selected_files]

    print(f"INFO: Opening {len(selected_files)} files with xarray/dask...")
    datasets = dask.compute(*delayed_datasets)
    
    ds = xr.concat(datasets, dim='time')
    
    return ds

In [8]:
# We pass all the dates to be loaded, as the files are monthly, very few dates would be dropped by the filters.

all_unique_dates = pd.concat([hw_dates['date'], bl_dates['date']]).unique()
all_dates_df = pd.DataFrame({'date': all_unique_dates})

# Reduced df for testing.
# all_dates_df = all_dates_df.sort_values(by='date').head(1000) #1000 = ~43 files

In [9]:
%%time

# So we load first, subset by lat/lon and compute. Here this reads and loads the u-component.

ds_u = get_days_delayed(all_dates_df, u_path, extent).persist()
wait(ds_u)

print('u components loaded and filtered.')

Building list of delayed tasks...
INFO: Opening 115 files with xarray/dask...
u components loaded and filtered.
CPU times: user 2min 58s, sys: 1min 22s, total: 4min 21s
Wall time: 7min 56s


In [ ]:
%%time

# Computing as above but for v-component files.

ds_v = get_days_delayed(all_dates_df, v_path, extent).persist()
wait(ds_v)

print('v components loaded and filtered.')

Building list of delayed tasks...
INFO: Opening 115 files with xarray/dask...


In [ ]:
# Merge u,v and display to look at chunk size.
ds_all = xr.merge([ds_u, ds_v])
ds_all

In [ ]:
# Now we convert to AEST to match NEM and filter for heatwave and baseline days.
# Define a fixed offset timezone for AEST (UTC+10, no daylight saving)
AEST = timezone(timedelta(hours=10))

hw_times_aest = pd.DatetimeIndex(hw_dates['date'].unique()).tz_localize("UTC").tz_convert(AEST).tz_localize(None)
bl_times_aest = pd.DatetimeIndex(bl_dates['date'].unique()).tz_localize("UTC").tz_convert(AEST).tz_localize(None)

ds_hw = ds_all.reindex(time=hw_times_aest, method=None)
ds_bl = ds_all.reindex(time=bl_times_aest, method=None)

In [ ]:
# Reshape to add hour and date components to our time coordinate (to bootstrap by day, for each hour).
ds_bl_reshaped = ds_bl.assign_coords(
    day=('time', ds_bl['time'].dt.floor('D').data),
    hour=('time', ds_bl['time'].dt.hour.data)
    ).set_index(time=['day', 'hour']).unstack('time')
ds_hw_reshaped = ds_hw.assign_coords(
    day=('time', ds_hw['time'].dt.floor('D').data),
    hour=('time', ds_hw['time'].dt.hour.data)
    ).set_index(time=['day', 'hour']).unstack('time')

ds_hw_chunked = ds_hw_reshaped.chunk({'day': 200, 'hour': -1, 'lat': -1, 'lon': -1})
ds_bl_chunked = ds_bl_reshaped.chunk({'day': 200, 'hour': -1, 'lat': -1, 'lon': -1})

In [ ]:
%%time
ds_hw = ds_hw_chunked.persist()
ds_bl = ds_bl_chunked.persist()

wait(ds_hw)
print('Heatwave input data ready for bootstrap')
wait(ds_bl)
print('Baseline input data ready for bootstrap')

ds_bl

In [ ]:
def bootstrap_on_days(data, n_boot, sample_size, seed=42):
    """
    Performs bootstrap resampling on the 'day' dimension of a given
    xarray Dataset or DataArray.

    This function is designed to be used with `groupby().apply()`.
    """
    resample_dim = 'day'
    n_days = data.sizes[resample_dim]
    if sample_size is None:
        sample_size = n_days

    rng = np.random.default_rng(seed)
    
    # Generate one set of random day indices for all variables in the dataset
    boot_idx = rng.integers(0, n_days, size=(n_boot, sample_size))
    boot_idx_da = xr.DataArray(boot_idx, dims=("boot", "sample"))

    # Select samples using the same indices for all data variables
    boot_samples = data.isel({resample_dim: boot_idx_da})
    
    # Return the mean composite for each bootstrap sample
    return boot_samples.mean(dim="sample")

In [ ]:
# Aiming for 1,000-10,000
n_boot = 1000
sample_size = 130

# This creates the lazy computation graph in Dask.
ds_hw_boot = ds_hw.groupby('hour').apply(
    bootstrap_on_days,
    n_boot=n_boot,
    sample_size=sample_size,
    seed=42  # Use a seed for reproducible results
    )

ds_bl_boot = ds_bl.groupby('hour').apply(
    bootstrap_on_days,
    n_boot=n_boot,
    sample_size=sample_size,
    seed=42  # Use the same seed for comparable distributions
    )

In [ ]:
hw_boot_results = ds_hw_boot.persist()
bl_boot_results = ds_bl_boot.persist()

wait(hw_boot_results)
print('Heatwave bootstrap composites calculated')
wait(bl_boot_results)
print('Baseline bootstrap composites calculated')

In [ ]:
# # This computes the composites of minimum, maximum, and diunal amplitude
# def compute_windspeed_composites(windspeed):
#     """
#     Compute windspeed composites (max, min, amplitude) from u and v.
    
#     Parameters
#     ----------
#     u, v : xarray.DataArray
#         Wind vector components (time x lat x lon)
    
#     Returns
#     -------
#     dict
#         {"max": DataArray, "min": DataArray, "diff": DataArray}
#     """
#     # Compute windspeed
    
#     # Compute composites along time dimension
#     composite_max = windspeed.max(dim="time").compute()
#     composite_min = windspeed.min(dim="time").compute()
#     diurnal_amp = (composite_max - composite_min).compute()
    
#     return composite_max, composite_min, diurnal_amp

# max_speed, min_speed, diurnal_amp = compute_windspeed_composites(ds_subset['windspeed'])

In [ ]:
# # --- Assemble into one Dataset ---
# results = xr.Dataset(
#     {
#         "windspeed_variance": hourly_var_per_point,
#         "windspeed_mean": hourly_composite["windspeed"],
#         "u_mean": hourly_composite["ua100m"],
#         "v_mean": hourly_composite["va100m"],
#         "windspeed_max": max_speed,
#         "windspeed_min": min_speed,
#         "diurnal_amp": diurnal_amp,
#     }
# )

# results = results.assign_attrs(reanalysis=reanalysis, mode=mode_str)

# # Write to NetCDF lazily with dask
# with ProgressBar():
#     results.to_netcdf(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc", compute=True)

In [ ]:
# xr.open_dataset(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc")